# Ежемесячные версии скоринговой модели

Оффер клиенту выбирает скоринговая модель — детерминированно: лучший по скору. Раз в месяц модель
переобучают, и каждая версия работает свой месяц. Вероятности выбора не логировались, но все версии
сохранены. Вопросы:

- какая версия лучшая, если сравнить их на одних и тех же клиентах;
- лучше ли таргетинг, чем случайная выдача с тем же балансом офферов — в целом и помесячно;
- можно ли по этим логам оценить новую модель;
- что изменить в проде, чтобы такие сравнения были возможны.

1. Данные
2. Почему одного месяца мало
3. Все месяцы вместе
4. Лучшая версия
5. Сравнение со случайной политикой
6. Новая модель
7. Случайный срез в проде
8. Сезонность
9. Чек-лист

Данные синтетические, поэтому каждый результат сверяется с истиной.

In [1]:
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

import policyscope as ps

SEED = 4                   # зерно симулятора
MONTHS = 12                # сколько месяцев (версий модели)
CLIENTS_PER_MONTH = 2_500  # клиентов в месяц
DRIFT = 0.35               # насколько версия меняется за одно переобучение
EXPLORATION = 0.10         # доля случайных показов в разделе 6

pd.set_option("display.float_format", lambda x: f"{x:,.0f}".replace(",", " ") if abs(x) >= 1e4 else f"{x:.4g}")
warnings.filterwarnings("ignore", message="Bootstrap:")


def values_vs_truth(board, policies, on, metric):
    """V каждого кандидата рядом с истиной на тех же клиентах (по одной метрике)."""
    rows = []
    for name, rep in board.reports.items():
        m = rep[metric]
        t = sim.true_value(on, policies[name], metric)
        rows.append({"кандидат": name, "оценка V": m.v_b.value, "ДИ от": m.v_b.low, "ДИ до": m.v_b.high,
                     "истина": t, "ДИ накрывает истину": "да" if m.v_b.low <= t <= m.v_b.high else "нет"})
    return pd.DataFrame(rows).set_index("кандидат")


def warnings_of(report):
    """Заголовок вывода и находки, которые снижают доверие."""
    icons = {"block": "⛔", "caution": "⚠️"}
    for metric, m in report.metrics.items():
        print(f"{metric}: {m.sentence(report.policy, report.baseline)}")
        for f in m.findings:
            if f.severity in icons:
                print(f"   {icons[f.severity]} {f.title}")

## 1. Данные

Что нужно от ваших данных:

- **логи**: признаки клиента на момент решения, показанный оффер, метрики и месяц (версия модели);
- **все версии модели** — чтобы пересчитать, что каждая версия рекомендовала бы любому клиенту.

В симуляторе `sim.versions` порождает версии, которые получаются при регулярном переобучении: каждая
следующая — предыдущая с небольшими изменениями.

In [2]:
sim = ps.synthetic.OfferSimulator(seed=SEED)
months = [f"2025-{m:02d}" for m in range(1, MONTHS + 1)]
versions = sim.versions(MONTHS, drift=DRIFT, seed=1, names=months)

clients = sim.sample_clients(MONTHS * CLIENTS_PER_MONTH)
rows_of = dict(zip(months, np.array_split(np.arange(len(clients)), MONTHS)))
history = pd.concat(
    [sim.simulate(clients.iloc[rows_of[m]], versions[m]).assign(month=m) for m in months],
    ignore_index=True,
).drop(columns="propensity")
history.head()

,client_id,age,income,loyal,risk,region,offer,accepted,revenue,month
0,0,24,73 000,0,0.966,west,insurance,0,0,2025-01
1,1,39,28 100,0,0.307,center,deposit,0,0,2025-01
2,2,74,136 000,1,0.219,east,card,1,158.4,2025-01
3,3,43,46 600,0,0.107,south,card,1,45.98,2025-01
4,4,67,83 700,1,0.88,center,insurance,1,93.52,2025-01


Насколько версии расходятся: сколько разных офферов клиенту рекомендовали бы 12 версий.

In [3]:
recommended = pd.DataFrame({m: versions[m].action_probs(history).idxmax(axis=1) for m in months})
recommended.nunique(axis=1).value_counts(normalize=True).sort_index().rename_axis(
    "разных офферов от всех версий").to_frame("доля клиентов")

,доля клиентов
разных офферов от всех версий,
1,0.4445
2,0.4245
3,0.1117
4,0.0193


Для клиентов, которым все версии рекомендуют одно и то же, логи знают отклик только на этот оффер.
Для остальных за год было исследование: оффер зависел от того, в каком месяце клиент пришёл.

## 2. Почему одного месяца мало

Одна версия детерминирована: каждому клиенту — один оффер. По логам одного месяца нельзя оценить
ни другую версию, ни случайную выдачу — вывод блокируется.

In [4]:
one_month = history[history["month"] == months[0]]
warnings_of(ps.evaluate(one_month, versions[months[1]], action="offer", reward="accepted",
                        logging_policy=versions[months[0]]))

accepted: вывод сделать нельзя: оценка Δ = −0.42 п.п. (−1.3%) ненадёжна (причины ниже).
   ⛔ Политика A почти не исследует
   ⚠️ IPS и DR расходятся


## 3. Все месяцы вместе

Передаём все логи, словарь версий и колонку месяца. Вероятность показать клиенту оффер — доля месяцев,
версии которых его рекомендуют (смесь версий с долями месяцев, balanced IPS). Пропенсити известны
точно, поэтому интервал аналитический.

Предпосылка — сопоставимые месяцы. Библиотека проверяет две вещи и пишет о них в отчёте:

- **состав клиентов** — угадывается ли месяц по признакам клиента;
- **отклик** — на клиентах, которым все версии рекомендуют одно и то же, отклик не должен зависеть от
  месяца (иначе это сезонность).

In [5]:
ev = ps.Evaluator(history, action="offer", reward=["accepted", "revenue"], logging_policy=versions, logger="month")
check = ev.evaluate(versions[months[-1]], name=months[-1])
print(next(f.detail for f in check.findings if f.code == "logging_mixture"))

Логи записаны 12 политиками (колонка «month»: 2025-01 — 8%, 2025-02 — 8%, 2025-03 — 8%, 2025-04 — 8%, 2025-05 — 8%, 2025-06 — 8%, 2025-07 — 8%, 2025-08 — 8%, 2025-09 — 8%, 2025-10 — 8%, 2025-11 — 8%, 2025-12 — 8%). Оценка несмещена, если клиенты и их отклик в периодах разных политик сопоставимы. Период по признакам клиента угадывается с ROC AUC 0.50: составы не различаются. Сезонность проверена на 13335 строках, где все политики выбирают одно и то же: отклик между периодами не различается.


## 4. Лучшая версия

Все версии оцениваются на всех клиентах сразу, а не каждая на своём месяце. База **A** — наблюдаемая
политика: каждый месяц работала своя версия. `best` выбирает лидера:

- набор лучших — кто значимо не уступает лидеру;
- одновременный интервал лидера — честный, хотя лидер выбран по тем же данным;
- ограничитель — доля принятых офферов не должна падать относительно A больше чем на 0.5 п.п.

In [6]:
board = ev.compare(versions)
selection = board.best("revenue", guardrails={"accepted": 0.005})
selection

- Лидер — «2025-12»: Δ к A = +8.42, одновременный 95% ДИ [+6.73; +10.11]. Этот интервал честен и после выбора лидера; точечная оценка лидера завышена (winner's curse).
- Лидер значимо лучше остальных допущенных кандидатов.
- Не допущен «2025-08»: не доказано, что «accepted» хуже, чем у A, не больше чем на 0.50 п.п.
- Не допущен «2025-03»: не доказано, что «accepted» хуже, чем у A, не больше чем на 0.50 п.п.
- Не допущен «2025-04»: не доказано, что «accepted» хуже, чем у A, не больше чем на 0.50 п.п.
- Не допущен «2025-01»: не доказано, что «accepted» хуже, чем у A, не больше чем на 0.50 п.п.
- Не допущен «2025-02»: не доказано, что «accepted» хуже, чем у A, не больше чем на 0.50 п.п.

Сверка с истиной. Для сравнения — «наивная» оценка: средняя выручка в месяце, когда версия работала.
Она видит версию только на её 2 500 клиентах, поэтому её интервал примерно вдвое шире; при разном
составе клиентов по месяцам она ещё и смещена.

In [7]:
table = values_vs_truth(board, versions, on=history, metric="revenue")
own = history.groupby("month")["revenue"].agg(["mean", "std", "count"])
table["наивно: среднее за свой месяц"] = own["mean"]
table["ширина ДИ: OPE"] = table["ДИ до"] - table["ДИ от"]
table["ширина ДИ: наивно"] = 2 * 1.96 * own["std"] / np.sqrt(own["count"])
table.sort_values("истина", ascending=False)

,оценка V,ДИ от,ДИ до,истина,ДИ накрывает истину,наивно: среднее за свой месяц,ширина ДИ: OPE,ширина ДИ: наивно
кандидат,,,,,,,,
2025-12,58.09,56.55,59.64,59.15,да,59.61,3.091,7.115
2025-06,55.91,54.59,57.23,56.41,да,57.71,2.638,6.772
2025-10,53.82,52.63,55,54.5,да,54.65,2.372,6.391
2025-11,54.4,53.18,55.61,54.49,да,53.7,2.432,6.404
2025-09,52.2,50.99,53.41,52.6,да,48.72,2.415,6.093
2025-05,51.91,50.84,52.98,51.95,да,54.67,2.147,6.47
2025-07,48.5,47.55,49.46,48.58,да,46.69,1.915,5.792
2025-08,47.77,46.79,48.75,48.12,да,47.83,1.959,5.929
2025-04,44.45,43.35,45.55,44.71,да,45.46,2.204,5.782


## 5. Сравнение со случайной политикой

База A — по-прежнему наблюдаемая политика. Кандидаты:

- `ps.random_like()` — случайная выдача с тем же балансом офферов, что за весь год;
- `ps.random_like(by="month")` — то же, но баланс сохраняется в каждом месяце отдельно;
- `ps.mixture(versions)` — «каждый раз случайная версия»: выбирает только из офферов, которые
  рекомендовали версии.

In [8]:
random_board = ev.compare({
    "случайная, баланс за год": ps.random_like(),
    "случайная, баланс по месяцам": ps.random_like(by="month"),
    "случайная версия": ps.mixture(versions),
})
for name, rep in random_board.reports.items():
    print(f"— {name}")
    warnings_of(rep)

— случайная версия
accepted: значимой разницы нет: Δ = −0.06 п.п. (−0.2%), ДИ [−0.15 п.п.; +0.03 п.п.]; эффект больше ±0.13 п.п. был бы обнаружен.
revenue: значимой разницы нет: Δ = −0.187 (−0.4%), ДИ [−0.44; +0.0652]; эффект больше ±0.361 был бы обнаружен.
— случайная, баланс за год
accepted: вывод сделать нельзя: оценка Δ = −8.02 п.п. (−22.2%) ненадёжна (причины ниже).
   ⛔ Политика выбирает действия там, где A их не выбирает
   ⚠️ IPS и DR расходятся
revenue: вывод сделать нельзя: оценка Δ = −5.92 (−11.9%) ненадёжна (причины ниже).
   ⛔ Политика выбирает действия там, где A их не выбирает
   ⚠️ IPS и DR расходятся
— случайная, баланс по месяцам
accepted: вывод сделать нельзя: оценка Δ = −8.08 п.п. (−22.4%) ненадёжна (причины ниже).
   ⛔ Политика выбирает действия там, где A их не выбирает
   ⚠️ IPS и DR расходятся
revenue: вывод сделать нельзя: оценка Δ = −6.01 (−12.1%) ненадёжна (причины ниже).
   ⛔ Политика выбирает действия там, где A их не выбирает
   ⚠️ IPS и DR расходятся


Случайная выдача заблокирована, и это правильно. Она назначает клиенту офферы, которых ему не
рекомендовала ни одна версия, а откликов на них в логах нет. Из таблицы раздела 1 видно, что все
четыре оффера за год встречались лишь у малой доли клиентов. Случайность нужна в логах, а не в
оцениваемой политике.

«Случайная версия» оценивается: она выбирает только из рекомендаций версий. Её значение близко к
наблюдаемому — так и должно быть, если месяцы сопоставимы.

## 6. Новая модель

Команда обучила новую модель отклика (на логах других клиентов за те же месяцы). Новая политика —
лучший оффер по ожидаемой выручке. По логам версий её можно оценить только там, где она выбирает
офферы, которые клиенту рекомендовала хотя бы одна версия. Логирующая политика известна, поэтому
отчёт знает эту долю точно.

In [9]:
train_clients = sim.sample_clients(MONTHS * 1_500)
train_rows = np.array_split(np.arange(len(train_clients)), MONTHS)
train = pd.concat([sim.simulate(train_clients.iloc[r], versions[m]) for m, r in zip(months, train_rows)],
                  ignore_index=True)
features = ["age", "income", "loyal", "risk", "region"]


def design(df, offer):
    X = pd.get_dummies(df[features], columns=["region"], dtype=float)
    for a in sim.actions:
        X[f"offer={a}"] = (offer == a).astype(float) if isinstance(offer, pd.Series) else float(offer == a)
    return X


response = HistGradientBoostingClassifier(random_state=0).fit(design(train, train["offer"]), train["accepted"])
avg_revenue = train[train["accepted"] == 1].groupby("offer")["revenue"].mean()


def new_scores(df):
    """Ожидаемая выручка каждого оффера по новой модели."""
    return pd.DataFrame({a: response.predict_proba(design(df, a))[:, 1] * avg_revenue[a] for a in sim.actions},
                        index=df.index)


new_model = ps.greedy(new_scores(history))
new_report = ev.evaluate(new_model, name="новая")
print(f"Решения новой модели вне покрытия логов: {new_report.overlap.uncovered:.0%}")
warnings_of(new_report)

Решения новой модели вне покрытия логов: 57%
accepted: вывод сделать нельзя: оценка Δ = +4.39 п.п. (+12.1%) ненадёжна (причины ниже).
   ⛔ Политика выбирает действия там, где A их не выбирает
   ⚠️ IPS и DR расходятся
revenue: вывод сделать нельзя: оценка Δ = +68.84 (+138.6%) ненадёжна (причины ниже).
   ⛔ Политика выбирает действия там, где A их не выбирает
   ⚠️ IPS и DR расходятся


Большая часть решений новой модели — офферы, которых версии этим клиентам не давали: откликов на них
в логах нет, и вывод заблокирован. Корректно по этим логам оценивается другая, более осторожная
политика: **новая модель в пределах версий** — лучший по её скорингу оффер среди тех, что клиенту уже
рекомендовала хоть одна версия (`allowed=ev.support()`). Она отвечает на вопрос, лучше ли новая
модель ранжирует уже используемые варианты, но не измеряет выигрыш от новых.

In [10]:
inside = ps.greedy(new_scores(history), allowed=ev.support())
inside_board = ev.compare({"новая в пределах версий": inside})
values_vs_truth(inside_board, {"новая в пределах версий": inside}, on=history, metric="revenue")

,оценка V,ДИ от,ДИ до,истина,ДИ накрывает истину
кандидат,,,,,
новая в пределах версий,63.57,61.67,65.47,65.29,да


## 7. Случайный срез в проде

Как сделать возможными сравнение со случайной выдачей и оценку новой модели целиком: в каждом месяце
небольшая доля показов (здесь 10%) — случайный оффер. Логирующая политика месяца — смесь версии и
равномерной выдачи; её вероятности известны точно, и пропенсити по-прежнему считать не нужно.

In [11]:
history_x = pd.concat(
    [sim.simulate(clients.iloc[rows_of[m]],
                  (1 - EXPLORATION) * versions[m].action_probs(clients.iloc[rows_of[m]]) + EXPLORATION / 4)
     .assign(month=m) for m in months],
    ignore_index=True,
).drop(columns="propensity")
logging_x = {m: ps.mixture([versions[m], ps.uniform()], weights=[1 - EXPLORATION, EXPLORATION]) for m in months}

ev_x = ps.Evaluator(history_x, action="offer", reward=["accepted", "revenue"], logging_policy=logging_x, logger="month")
best_version = selection.winner
new_model_x = ps.greedy(new_scores(history_x))
board_x = ev_x.compare({
    f"лучшая версия ({best_version})": versions[best_version],
    "новая модель": new_model_x,
    "случайная, баланс за год": ps.random_like(),
    "случайная, баланс по месяцам": ps.random_like(by="month"),
})
board_x

## Сравнение кандидатов с A

| Кандидат | Метрика | V(A) [95% ДИ] | V(кандидата) [95% ДИ] | Δ | 95% ДИ для Δ | Δ, % | p | p (RW) | Вывод | Доверие | Оценщики |
|---|---|---|---|---|---|---|---|---|---|---|---|
| новая модель | accepted | 35.14% [34.60%; 35.68%] | 42.87% [40.35%; 45.39%] | +7.73 п.п. | [+5.26 п.п.; +10.19 п.п.] | +22.0% | <0.001 | <0.001 | лучше A | с осторожностью | согласованы |
| новая модель | revenue | 50.20 [49.26; 51.14] | 117.9 [107.8; 128.0] | +67.71 | [+57.83; +77.59] | +134.9% | <0.001 | <0.001 | лучше A | с осторожностью | согласованы |
| лучшая версия (2025-12) | accepted | 35.14% [34.60%; 35.68%] | 38.37% [37.58%; 39.15%] | +3.22 п.п. | [+2.65 п.п.; +3.79 п.п.] | +9.2% | <0.001 | <0.001 | лучше A | надёжно | согласованы |
| лучшая версия (2025-12) | revenue | 50.20 [49.26; 51.14] | 59.60 [58.03; 61.16] | +9.40 | [+8.18; +10.62] | +18.7% | <0.001 | <0.001 | лучше A | надёжно | согласованы |
| случайная, баланс за год | accepted | 35.14% [34.60%; 35.68%] | 27.82% [26.76%; 28.88%] | −7.33 п.п. | [−8.31 п.п.; −6.35 п.п.] | −20.8% | <0.001 | <0.001 | хуже A | надёжно | согласованы |
| случайная, баланс за год | revenue | 50.20 [49.26; 51.14] | 43.23 [41.47; 44.99] | −6.97 | [−8.57; −5.37] | −13.9% | <0.001 | <0.001 | хуже A | надёжно | согласованы |
| случайная, баланс по месяцам | accepted | 35.14% [34.60%; 35.68%] | 27.63% [26.55%; 28.70%] | −7.52 п.п. | [−8.51 п.п.; −6.53 п.п.] | −21.4% | <0.001 | <0.001 | хуже A | надёжно | согласованы |
| случайная, баланс по месяцам | revenue | 50.20 [49.26; 51.14] | 42.90 [41.08; 44.71] | −7.30 | [−8.94; −5.66] | −14.5% | <0.001 | <0.001 | хуже A | надёжно | согласованы |

p (RW) скорректированы методом Romano–Wolf на 8 пар «кандидат × метрика».

### Кто лучший по «accepted»
- Лидер — «новая модель»: Δ к A = +7.73 п.п., одновременный 95% ДИ [+4.76 п.п.; +10.69 п.п.]. Этот интервал честен и после выбора лидера; точечная оценка лидера завышена (winner's curse).
- Лидер значимо лучше остальных допущенных кандидатов.

Лидер против остальных (одновременные 95% ДИ по всем парам кандидатов, p с поправкой Romano–Wolf; все пары — в `board.pairwise`):

| Против | Разница «новая модель» − кандидат | 95% ДИ | p (RW) |
|---|---|---|---|
| лучшая версия (2025-12) | +4.50 п.п. | [+1.21 п.п.; +7.80 п.п.] | 0.002 |
| случайная, баланс за год | +15.05 п.п. | [+11.93 п.п.; +18.17 п.п.] | <0.001 |
| случайная, баланс по месяцам | +15.24 п.п. | [+12.12 п.п.; +18.37 п.п.] | <0.001 |

Другая метрика выбора или ограничители — `board.best(metric, guardrails={...})`.

In [12]:
year_shares = history_x["offer"].value_counts(normalize=True)
month_shares = history_x.groupby("month")["offer"].value_counts(normalize=True).unstack(fill_value=0)
values_vs_truth(board_x, on=history_x, metric="revenue", policies={
    f"лучшая версия ({best_version})": versions[best_version],
    "новая модель": new_model_x,
    "случайная, баланс за год": pd.DataFrame(np.tile(year_shares.reindex(sim.actions).to_numpy(), (len(history_x), 1)),
                                             index=history_x.index, columns=sim.actions),
    "случайная, баланс по месяцам": month_shares.reindex(columns=sim.actions, fill_value=0)
                                    .loc[history_x["month"]].set_axis(history_x.index),
})

,оценка V,ДИ от,ДИ до,истина,ДИ накрывает истину
кандидат,,,,,
новая модель,117.9,107.8,128,111.8,да
лучшая версия (2025-12),59.6,58.03,61.16,59.15,да
"случайная, баланс за год",43.23,41.47,44.99,42.78,да
"случайная, баланс по месяцам",42.9,41.08,44.71,42.77,да


**Какой срез нужен.** Решения новой модели вне покрытия версий оцениваются только по строкам среза.
Эффективный размер выборки прикидывается заранее: $ESS \approx n \cdot \varepsilon / (k \cdot u)$,
где $\varepsilon$ — доля среза, $k$ — число офферов, $u$ — доля решений модели вне покрытия
(из раздела 6). Для надёжной оценки нужны хотя бы тысячи эффективных строк.

In [13]:
u = new_report.overlap.uncovered
planned = len(history_x) * EXPLORATION / (len(sim.actions) * u)
print(f"Прогноз ESS: {planned:,.0f}; по факту: {board_x['новая модель'].overlap.ess:,.0f}".replace(",", " "))

Прогноз ESS: 1 313; по факту: 1 240


## 8. Сезонность

Главный риск помесячной схемы — месяцы различаются не только версией модели. Смоделируем декабрь, в
котором клиенты откликаются заметно охотнее. Отчёт заметит это на клиентах, которым все версии
рекомендуют одно и то же.

In [14]:
seasonal = pd.concat(
    [sim.simulate(clients.iloc[rows_of[m]], versions[m], shift=0.6 if m.endswith("-12") else 0.0).assign(month=m)
     for m in months],
    ignore_index=True,
).drop(columns="propensity")
seasonal_report = ps.evaluate(seasonal, versions[months[0]], action="offer", reward=["accepted", "revenue"],
                              logging_policy=versions, logger="month")
print(next(f.detail for f in seasonal_report.findings if f.code == "logger_drift"))

На 13335 строках, где все политики выбирают одно и то же, средний отклик «accepted» с поправкой на признаки различается между периодами на 0.152 (p = 4.2e-16). Похоже на сезонность или изменения продукта: смесь периодов смешивает и эти различия.


В таком случае смесь месяцев смешивает и сезонный эффект. Сравнивайте соседние, сопоставимые месяцы
или исключите аномальные.

## 9. Чек-лист

1. Сохраняйте каждую версию модели вместе с правилом выбора оффера.
2. Логируйте месяц (версию), показанный оффер и признаки клиента на момент решения.
3. Держите небольшой случайный срез, 5–10% показов: с ним можно сравнивать что угодно, в том числе
   со случайной выдачей, а без него — только версии и их смеси.
4. Сравнивайте сопоставимые месяцы и смотрите на находки о составе клиентов и сезонности.
5. Лучшую версию выбирайте через `board.best(...)`. Если в наборе лучших несколько версий, нужны
   свежие данные или A/B-тест.
6. Новую модель по логам версий можно оценить только там, где она выбирает офферы, которые давали
   версии: смотрите «вне покрытия логов» в отчёте и оценивайте вариант `allowed=ev.support()`. Для
   оценки целиком нужен случайный срез; его размер прикиньте по формуле из раздела 7.

Общий разбор библиотеки — в [tutorial.ipynb](tutorial.ipynb), методология — в
[docs/methodology.md](../docs/methodology.md).